# ARC-Hydra ARC-AGI-2 CPU starting baseline

A bounded demonstration-consistent symbolic program search, separate from ARC-AGI-3. Tries geometry, color mappings, foreground/object crops, scaling, tiling and split-grid combinations. Two distinct predictions where available; explicit input-copy fallback otherwise. Uses only each task’s supplied demonstrations, never evaluation-answer files or task-ID lookup. A reproducible floor, not a claim of competitive performance. No GPU or model training. Local public-training measurement: 49/1076 test outputs (4.55%); not an unseen or official result.

In [ ]:
from pathlib import Path
import json,time,sys
sources={'arc2/__init__.py': '"""Separate ARC-AGI-2 experiments; no dependency on the ARC-AGI-3 runtime."""\n', 'arc2/symbolic.py': '"""Bounded task-local program search; uses only the task\'s demonstration pairs."""\nfrom collections import Counter,deque\nimport time\n\n\ndef grid(value):\n    if not isinstance(value,(list,tuple)) or not 1<=len(value)<=30:raise ValueError(\'Invalid grid height\')\n    rows=tuple(tuple(row) for row in value)\n    if not rows[0] or len(rows[0])>30 or any(len(row)!=len(rows[0]) for row in rows):raise ValueError(\'Invalid grid width\')\n    if any(type(c) is not int or not 0<=c<=9 for row in rows for c in row):raise ValueError(\'Invalid grid colors\')\n    return rows\n\n\ndef modal(g):return Counter(c for row in g for c in row).most_common(1)[0][0]\n\n\ndef rotate(g):return tuple(tuple(row) for row in zip(*g[::-1]))\n\n\ndef orient(g,index):\n    if index>=4:g=tuple(tuple(row) for row in zip(*g));index-=4\n    for _ in range(index):g=rotate(g)\n    return g\n\n\ndef crop(g,color=None,background=None):\n    bg=modal(g) if background is None else background\n    points=[(r,c) for r,row in enumerate(g) for c,x in enumerate(row) if (x==color if color is not None else x!=bg)]\n    if not points:return None\n    r0=min(p[0] for p in points);r1=max(p[0] for p in points)+1\n    c0=min(p[1] for p in points);c1=max(p[1] for p in points)+1\n    return tuple(row[c0:c1] for row in g[r0:r1])\n\n\ndef object_crop(g,selection,monochrome,background):\n    bg=modal(g) if background is None else background\n    seen=set();objects=[];h=len(g);w=len(g[0])\n    for r in range(h):\n        for c in range(w):\n            if (r,c) in seen or g[r][c]==bg:continue\n            pending=deque([(r,c)]);seen.add((r,c));points=[]\n            while pending:\n                y,x=pending.popleft();points.append((y,x))\n                for yy,xx in ((y-1,x),(y+1,x),(y,x-1),(y,x+1)):\n                    if 0<=yy<h and 0<=xx<w and (yy,xx) not in seen and g[yy][xx]!=bg and (not monochrome or g[yy][xx]==g[r][c]):\n                        seen.add((yy,xx));pending.append((yy,xx))\n            objects.append(points)\n    if not objects:return None\n    objects.sort(key=lambda p:(len(p),min(p)))\n    points=objects[-1] if selection==\'largest\' else objects[0]\n    r0=min(p[0] for p in points);r1=max(p[0] for p in points)+1\n    c0=min(p[1] for p in points);c1=max(p[1] for p in points)+1\n    return tuple(row[c0:c1] for row in g[r0:r1])\n\n\ndef resize(g,scale,repeat=False):\n    if len(g)*scale>30 or len(g[0])*scale>30:return None\n    if repeat:return tuple(row*scale for _ in range(scale) for row in g)\n    return tuple(tuple(c for x in row for c in [x]*scale) for row in g for _ in range(scale))\n\n\ndef reduce_grid(g,scale):\n    if len(g)%scale or len(g[0])%scale:return None\n    result=[]\n    for r in range(0,len(g),scale):\n        row=[]\n        for c in range(0,len(g[0]),scale):\n            values={g[y][x] for y in range(r,r+scale) for x in range(c,c+scale)}\n            if len(values)!=1:return None\n            row.append(next(iter(values)))\n        result.append(tuple(row))\n    return tuple(result)\n\n\ndef compress(g):\n    rows=[]\n    for row in g:\n        if not rows or row!=rows[-1]:rows.append(row)\n    columns=[]\n    for col in zip(*rows):\n        if not columns or col!=columns[-1]:columns.append(col)\n    return tuple(tuple(row) for row in zip(*columns))\n\n\ndef halves(g,axis,mode,background):\n    if axis:g=orient(g,4)\n    n=len(g);middle=n//2\n    if n<2:return None\n    first=g[:middle];second=g[middle+n%2:]\n    bg=modal(g) if background is None else background\n    if mode==\'first\':out=first\n    elif mode==\'second\':out=second\n    else:\n        operations={\'or\':lambda a,b:a or b,\'and\':lambda a,b:a and b,\'xor\':lambda a,b:a!=b}\n        op=operations[mode]\n        out=tuple(tuple(int(op(a!=bg,b!=bg)) for a,b in zip(ra,rb)) for ra,rb in zip(first,second))\n    return orient(out,4) if axis else out\n\n\ndef operations(colors):\n    yield \'identity\',0,lambda g:g\n    yield \'compress_repeated_rows_columns\',1,compress\n    for bg in (None,0):\n        yield f\'foreground_bbox_{bg}\',1,lambda g,bg=bg:crop(g,background=bg)\n        for selection in (\'largest\',\'smallest\'):\n            for mono in (False,True):\n                yield f\'{selection}_object_{mono}_{bg}\',2,lambda g,s=selection,m=mono,b=bg:object_crop(g,s,m,b)\n        for axis in (0,1):\n            for mode in (\'first\',\'second\',\'or\',\'and\',\'xor\'):\n                yield f\'halves_{axis}_{mode}_{bg}\',2,lambda g,a=axis,m=mode,b=bg:halves(g,a,m,b)\n    for color in colors:yield f\'color_bbox_{color}\',2,lambda g,c=color:crop(g,color=c)\n    for scale in (2,3,4):\n        yield f\'upscale_{scale}\',1,lambda g,s=scale:resize(g,s)\n        yield f\'tile_{scale}\',2,lambda g,s=scale:resize(g,s,True)\n        yield f\'uniform_downscale_{scale}\',1,lambda g,s=scale:reduce_grid(g,s)\n\n\ndef fit_mapping(inputs,outputs):\n    mapping={}\n    for before,after in zip(inputs,outputs):\n        if before is None or len(before)!=len(after) or len(before[0])!=len(after[0]):return None\n        for ra,rb in zip(before,after):\n            for a,b in zip(ra,rb):\n                if a in mapping and mapping[a]!=b:return None\n                mapping[a]=b\n    return mapping\n\n\ndef solve(task,seconds=1.0):\n    started=time.monotonic();examples=task[\'train\'];tests=[grid(t[\'input\']) for t in task[\'test\']]\n    if not examples:return [dict(attempt_1=[list(row) for row in g],attempt_2=[list(row) for row in g]) for g in tests],dict(programs=0,reason=\'no demonstrations\')\n    inputs=[grid(t[\'input\']) for t in examples];outputs=[grid(t[\'output\']) for t in examples]\n    colors=sorted({c for g in inputs for row in g for c in row})\n    mappings={};prediction_sets=set();candidates=[];tried=0;timed_out=False\n    for name,cost,operation in operations(colors):\n        if time.monotonic()-started>=seconds:timed_out=True;break\n        transformed=[operation(g) for g in inputs]\n        if any(g is None for g in transformed):continue\n        for orientation in range(8):\n            views=tuple(orient(g,orientation) for g in transformed)\n            tried+=1\n            if views not in mappings:mappings[views]=fit_mapping(views,outputs)\n            mapping=mappings[views]\n            if mapping is None:continue\n            predictions=[]\n            for g in tests:\n                result=operation(g)\n                if result is None:break\n                result=orient(result,orientation)\n                result=tuple(tuple(mapping.get(c,c) for c in row) for row in result)\n                predictions.append(result)\n            if len(predictions)!=len(tests):continue\n            signature=tuple(predictions)\n            if signature in prediction_sets:continue\n            prediction_sets.add(signature)\n            recolors=sum(a!=b for a,b in mapping.items())\n            score=cost+0.1*(orientation!=0)+0.05*recolors\n            candidates.append((score,name,orientation,tuple(predictions)))\n    candidates.sort(key=lambda c:(c[0],c[1],c[2]))\n    attempts=[]\n    for i,g in enumerate(tests):\n        unique=[]\n        for candidate in candidates:\n            prediction=candidate[3][i]\n            if prediction not in unique:unique.append(prediction)\n            if len(unique)==2:break\n        if not unique:unique=[g]\n        if len(unique)==1:unique.append(g if g!=unique[0] else tuple(tuple(0 for _ in row) for row in g))\n        attempts.append(dict(attempt_1=[list(row) for row in unique[0]],attempt_2=[list(row) for row in unique[1]]))\n    return attempts,dict(programs=tried,consistent_programs=len(candidates),timed_out=timed_out,\n        elapsed_seconds=time.monotonic()-started,selected=[dict(name=c[1],orientation=c[2],complexity=c[0]) for c in candidates[:2]])\n', 'arc2/evaluation.py': '"""Official two-attempt exact-match metric, averaged over test outputs."""\nfrom .symbolic import grid\n\n\ndef validate_submission(challenges,submission):\n    if set(challenges)!=set(submission):raise ValueError(\'Submission IDs must exactly match challenge IDs\')\n    for key,task in challenges.items():\n        guesses=submission[key]\n        if not isinstance(guesses,list) or len(guesses)!=len(task[\'test\']):raise ValueError(\'Test-output count mismatch: \'+key)\n        for attempt in guesses:\n            if not isinstance(attempt,dict) or set(attempt)!=set((\'attempt_1\',\'attempt_2\')):\n                raise ValueError(\'Exactly two named attempts required\')\n            grid(attempt[\'attempt_1\']);grid(attempt[\'attempt_2\'])\n\n\ndef score_submission(challenges,solutions,submission):\n    validate_submission(challenges,submission)\n    if set(challenges)!=set(solutions):raise ValueError(\'Solution IDs differ\')\n    results=[]\n    for key,task in challenges.items():\n        if len(solutions[key])!=len(task[\'test\']):raise ValueError(\'Solution-output count mismatch\')\n        for index,target in enumerate(solutions[key]):\n            target=grid(target);attempt=submission[key][index]\n            correct=any(grid(attempt[name])==target for name in (\'attempt_1\',\'attempt_2\'))\n            results.append(dict(task_id=key,test_index=index,correct=correct))\n    return dict(correct_outputs=sum(r[\'correct\'] for r in results),test_outputs=len(results),\n        score=100*sum(r[\'correct\'] for r in results)/len(results) if results else 0,per_output=results)\n'}
for name,source in sources.items():
    path=Path('/kaggle/working')/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(source)
sys.path.insert(0,'/kaggle/working')
from arc2.symbolic import solve
from arc2.evaluation import validate_submission
roots=[Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2'),Path('/kaggle/input/arc-prize-2026-arc-agi-2')]
root=next((p for p in roots if (p/'arc-agi_test_challenges.json').is_file()),None)
if root is None:raise FileNotFoundError('ARC-AGI-2 input challenges not mounted')
challenges=json.loads((root/'arc-agi_test_challenges.json').read_text())
started=time.monotonic();submission={};events=[]
for key,task in sorted(challenges.items()):
    if time.monotonic()-started>3500:
        guesses=[dict(attempt_1=t['input'],attempt_2=t['input']) for t in task['test']]
        info=dict(reason='global budget fallback',consistent_programs=0)
    else:guesses,info=solve(task,seconds=2.0)
    submission[key]=guesses;events.append(dict(task_id=key,**info))
validate_submission(challenges,submission)
Path('/kaggle/working/submission.json').write_text(json.dumps(submission))
report=dict(tasks=len(challenges),test_outputs=sum(len(t['test']) for t in challenges.values()),
    elapsed_seconds=time.monotonic()-started,tasks_with_consistent_program=sum(e.get('consistent_programs',0)>0 for e in events),
    boundary='Symbolic CPU starting baseline; no neural model, weight training or official score established.')
Path('/kaggle/working/arc2-run-metrics.json').write_text(json.dumps(report,indent=2))
Path('/kaggle/working/arc2-search-events.json').write_text(json.dumps(events))
print('ARC2_CPU_BASELINE_COMPLETE',json.dumps(report),flush=True)
